# 1. Konfiguracja środowiska oraz datasetu

## 1.1. Instalacja zależności

In [ ]:
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#%pip install -r requirements.txt

## 1.2.Konfiguracja importów

In [ ]:
import os
import shutil
from pathlib import Path

import torch
from torch import optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from tqdm import tqdm

from v1.config import ANNOTATION_HOP, NOTES_BINS_PER_SEMITONE, GUITAR_BASE_FREQUENCY
from v1.config import N_FREQ_BINS_NOTES, N_FREQ_BINS_CONTOURS
from v1.dataset import GuitarSetDataset, BucketBatchSampler, collate_fn
from v1.losses import calculate_pos_weights_for_dataset
from v1.midi_conversion import predictions_to_midi
from v1.process_guitarset import process_dataset
from v1.train import evaluate_epoch
from v1.train import train_model
from v1.tresholds import gather_predictions_and_targets, find_optimal_thresholds
from v1.validation import batch_validate
from v1.visualization import plot_sample_data

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
torch.cuda.empty_cache()

## 1.3. Konfiguracja GPU - automatyczne wykrywanie

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")
if device.type == 'cuda':
    print(f"Model karty GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

## 1.4. Wczytanie guitarsetu

In [ ]:
config = {
    "data_dir": "v1/guitarset_data",
    "output_dir": "v1/processed_data",
    "seed": 42,
    "max_tracks": None,
    "overwrite": False
}

process_dataset(**config)

## 1.5. Wczytanie guitarseta z plików batch

In [ ]:
# Wczytanie danych z plików
train_dir = Path(config["output_dir"]) / "train"
val_dir = Path(config["output_dir"]) / "val"
test_dir = Path(config["output_dir"]) / "test"

# Utworzenie datasetów
train_dataset = GuitarSetDataset(
    train_dir,
    is_train_dataset=True,
    augment_onsets_prob=0.75,
    onset_blur_window_size=1
)
val_dataset = GuitarSetDataset(val_dir, is_train_dataset=False)  # Bez augmentacji dla walidacji
test_dataset = GuitarSetDataset(test_dir, is_train_dataset=False)  # Bez augmentacji dla testu

use_cuda = torch.cuda.is_available()
pin_memory = use_cuda
optimal_num_workers = 6

# Stworzenie samplera dla zbioru treningowego
train_batch_sampler = BucketBatchSampler(
    length_cache=train_dataset.lengths_cache,
    num_samples=len(train_dataset),
    batch_size=1,
    shuffle=True,
    drop_last=True
)

# Aktualizacja DataLoaderów
train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_batch_sampler,
    collate_fn=collate_fn,
    num_workers=optimal_num_workers,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=optimal_num_workers,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=optimal_num_workers,
    pin_memory=pin_memory
)

print("DataLoadery skonfigurowane.")
print(f"Używam GPU: {use_cuda}, pin_memory: {pin_memory}")
print(f"Liczba workerów: {optimal_num_workers}")

## 1.6. Walidacja batcha


In [ ]:
batch_validate(train_loader)

## 1.7. Wizualizacja przykładowego pliku

In [ ]:
sample_batch = next(iter(train_loader))
plot_sample_data(sample_batch, sample_idx=1)

# 2. Model i uczenie

## 2.1. Wczytanie implementacji modelu

In [ ]:
from v1.model import MultiTaskTranscriptionModel

model = MultiTaskTranscriptionModel().to(device)

## 2.2. Konfiguracja treningu

In [ ]:
# Typ funkcji straty: "Focal" lub "BCE"
LOSS_FUNCTION_TYPE = "Focal"

# Wagi dla poszczególnych zadań
TASK_WEIGHTS = {
    "notes": 0.8,
    "onsets": 40.0,
    "contours": 1.1,
}

POS_WEIGHTS_FOR_LOSS = calculate_pos_weights_for_dataset(
    train_loader,
    device,
    N_FREQ_BINS_NOTES,
    N_FREQ_BINS_CONTOURS,
)

FOCAL_LOSS_GAMMA = 4.0
FOCAL_LOSS_ALPHA_CONFIG = {
    "notes": 0.65,
    "onsets": 0.95,
    "contours": 0.8,
}

# Hiperparametry treningu
LEARNING_RATE = 1e-4
NUM_EPOCHS = 1000
WEIGHT_DECAY = 1e-3

# Scheduler ReduceLROnPlateau
SCHEDULER_PATIENCE = 15
SCHEDULER_FACTOR = 0.5

# Gradient clipping
GRADIENT_CLIP_VAL = 1.1

# Ścieżki i nazwy plików
CHECKPOINT_DIR = "v1/checkpoints_focal_aug"
MODEL_NAME = "best_model.pt"

# Early stopping
EARLY_STOPPING_PATIENCE = 35
METRIC_FOR_CHECKPOINT = "val_onsets_f1"

# Optymalizator
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler_mode = "max"
if "loss" in METRIC_FOR_CHECKPOINT:
    scheduler_mode = "min"

scheduler = ReduceLROnPlateau(
    optimizer,
    mode=scheduler_mode,
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
)


## 2.3. Trening modelu

In [ ]:
training_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    device=device,
    pos_weights=POS_WEIGHTS_FOR_LOSS,
    task_weights=TASK_WEIGHTS,
    gradient_clip_val=GRADIENT_CLIP_VAL,
    loss_type=LOSS_FUNCTION_TYPE,
    focal_loss_gamma=FOCAL_LOSS_GAMMA,
    focal_loss_alpha=FOCAL_LOSS_ALPHA_CONFIG,
    checkpoint_dir=CHECKPOINT_DIR,
    model_name=MODEL_NAME,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    metric_for_checkpoint=METRIC_FOR_CHECKPOINT,
    plot_every_n_epochs=10
)

print("\nHistoria treningu (średnie straty i metryki na zbiorze walidacyjnym):")
if 'train_loss_total' in training_history and training_history['train_loss_total']:
    num_actual_epochs = len(training_history['train_loss_total'])
    print(f"Trening trwał {num_actual_epochs} epok.")
    print(f"Ostatnia strata treningowa (całkowita): {training_history['train_loss_total'][-1]:.4f}")
else:
    print("Brak danych o stratach treningowych w historii lub trening nie wygenerował żadnej epoki.")
    num_actual_epochs = 0

if num_actual_epochs > 0:
    if 'val_loss_total' in training_history and training_history['val_loss_total']:
        print(f"Ostatnia strata walidacyjna (całkowita): {training_history['val_loss_total'][-1]:.4f}")
    else:
        print("Brak danych o stratach walidacyjnych w historii.")

    print(f"\nOstatnie metryki walidacyjne (dla epoki {num_actual_epochs}):")
    # Użyj .get() z domyślną wartością na wypadek, gdyby klucze nie istniały
    notes_f1 = training_history.get('val_notes_f1', [float('nan')])[-1]
    notes_p = training_history.get('val_notes_precision', [float('nan')])[-1]
    notes_r = training_history.get('val_notes_recall', [float('nan')])[-1]
    notes_auc_pr = training_history.get('val_notes_auc_pr', [float('nan')])[-1]

    onsets_f1 = training_history.get('val_onsets_f1', [float('nan')])[-1]
    onsets_p = training_history.get('val_onsets_precision', [float('nan')])[-1]
    onsets_r = training_history.get('val_onsets_recall', [float('nan')])[-1]
    onsets_auc_pr = training_history.get('val_onsets_auc_pr', [float('nan')])[-1]

    contours_f1 = training_history.get('val_contours_f1', [float('nan')])[-1]
    contours_p = training_history.get('val_contours_precision', [float('nan')])[-1]
    contours_r = training_history.get('val_contours_recall', [float('nan')])[-1]
    contours_auc_pr = training_history.get('val_contours_auc_pr', [float('nan')])[-1]

    print(f"  Notes:    F1={notes_f1:.4f} (P={notes_p:.4f}, R={notes_r:.4f}, AUC-PR={notes_auc_pr:.4f})")
    print(f"  Onsets:   F1={onsets_f1:.4f} (P={onsets_p:.4f}, R={onsets_r:.4f}, AUC-PR={onsets_auc_pr:.4f})")
    print(
        f"  Contours: F1={contours_f1:.4f} (P={contours_p:.4f}, R={contours_r:.4f}, AUC-PR={contours_auc_pr:.4f})")

elif num_actual_epochs == 0 and 'val_onsets_f1' in training_history:  # Możliwe, że była tylko jedna ewaluacja
    print("Trening nie wygenerował żadnej pełnej epoki, ale mogą istnieć początkowe metryki walidacyjne.")
    # Wyświetl metryki jeśli są
    if training_history.get('val_onsets_f1'):  # Sprawdź czy lista nie jest pusta
        onsets_f1 = training_history.get('val_onsets_f1', [float('nan')])[-1]
        print(f"  Onsets F1 (początkowe): {onsets_f1:.4f}")



## 2.4. Obliczanie treshold

In [ ]:
best_model_path = Path(CHECKPOINT_DIR) / MODEL_NAME
if best_model_path.exists():
    model.load_state_dict(torch.load(best_model_path,
                                     map_location=device, weights_only=True))
    print(f"Załadowano model z: {best_model_path}")
else:
    print(f"OSTRZEŻENIE: Nie znaleziono pliku modelu: {best_model_path}. Używam modelu z ostatniego stanu.")

model.to(device)
model.eval()

# Upewnij się, że val_loader nie jest pusty
if len(val_loader) > 0:
    all_y_true_val, all_y_scores_val = gather_predictions_and_targets(model, val_loader, device)
    optimal_thresholds = find_optimal_thresholds(all_y_true_val, all_y_scores_val, plot_curves=True)
    print("\nAutomatycznie znalezione optymalne progi (maksymalizujące F1 na walidacji):")
    print(optimal_thresholds)
else:
    print("val_loader jest pusty. Nie można obliczyć optymalnych progów.")
    optimal_thresholds = {"notes": 0.5, "onsets": 0.5, "contours": 0.5}  # Domyślne wartości


## 2.5. Ewaluacja modelu na danych testowych.

In [ ]:
if len(test_loader) > 0:
    print("\nEwaluacja na zbiorze testowym z optymalnymi progami:")
    test_results_optimized_thresholds = evaluate_epoch(
        model=model,
        dataloader=test_loader,
        device=device,
        pos_weights=POS_WEIGHTS_FOR_LOSS,
        task_weights=TASK_WEIGHTS,
        epoch_num=0,
        checkpoint_dir=str(Path(CHECKPOINT_DIR) / "test_results"),
        model_name_stem=MODEL_NAME.replace(".pt", "") + "_test_final_optimized_thresholds",
        loss_type=LOSS_FUNCTION_TYPE,
        focal_loss_gamma=FOCAL_LOSS_GAMMA,
        focal_loss_alpha=FOCAL_LOSS_ALPHA_CONFIG,
        plot_every_n_epochs=0,
        sample_to_plot_idx=0,
        thresholds=optimal_thresholds
    )

    metrics = test_results_optimized_thresholds['metrics']
    loss_info = test_results_optimized_thresholds['loss']

    print(
        f"\n--- Wyniki ewaluacji na zbiorze testowym (progi: N: {optimal_thresholds.get('notes', 0.5):.2f}, O: {optimal_thresholds.get('onsets', 0.5):.2f}, C: {optimal_thresholds.get('contours', 0.5):.2f}) ---")
    print(f"  Użyta funkcja straty podczas tej ewaluacji: {LOSS_FUNCTION_TYPE.upper()}")
    print(f"  Strata całkowita: {loss_info['total_loss']:.4f}")
    print(f"    Strata Notes:    {loss_info['loss_notes']:.4f}")
    print(f"    Strata Onsets:   {loss_info['loss_onsets']:.4f}")
    print(f"    Strata Contours: {loss_info['loss_contours']:.4f}")
    print("-" * 40)

    tasks = ["notes", "onsets", "contours"]
    metric_names_display = {
        "precision": "Precision",
        "recall": "Recall   ",  # Dodano spacje dla wyrównania
        "f1": "F1-score ",  # Dodano spacje dla wyrównania
        "auc_pr": "AUC-PR   "  # Dodano spacje dla wyrównania
    }

    for task in tasks:
        print(f"  {task.capitalize()}:")
        for metric_key, display_name in metric_names_display.items():
            # Użyj .get() z domyślną wartością, jeśli klucz nie istnieje
            value = metrics.get(f"{task}_{metric_key}", float('nan'))
            print(f"    {display_name}: {value:.4f}")
        print("-" * 20)
else:
    print("test_loader jest pusty. Nie można przeprowadzić ewaluacji na zbiorze testowym.")


## 3. Konwersja testowego audio do formatu midi

In [ ]:
NUM_SAMPLES_TO_TRANSCRIBE = 3
MIDI_OUTPUT_DIR = Path("v1/transcribed_examples_from_test_loader")
MIDI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GUITARSET_ORIGINAL_AUDIO_DIR = Path(config["data_dir"]) / "audio_mono-mic"

model.eval()

print(f"\nTranskrypcja {NUM_SAMPLES_TO_TRANSCRIBE} próbek ze zbioru testowego do MIDI...")

transcribed_count = 0
for batch_test in tqdm(test_loader, desc="Transkrypcja próbek testowych"):
    if transcribed_count >= NUM_SAMPLES_TO_TRANSCRIBE:
        break

    features_batch = batch_test["features"].to(device)
    feature_lengths_batch = batch_test["feature_lengths"]
    track_ids_batch = batch_test.get("track_ids")

    if track_ids_batch is None:
        print(
            "OSTRZEŻENIE: Brak 'track_ids' w batchu. Upewnij się, że process_dataset i collate_fn zostały zaktualizowane.")
        print("Przerywanie transkrypcji MIDI.")
        break

    with torch.no_grad():
        output_logits_batch = model(features_batch)

    for i in range(features_batch.shape[0]):
        if transcribed_count >= NUM_SAMPLES_TO_TRANSCRIBE:
            break

        track_id = track_ids_batch[i]

        original_audio_path = GUITARSET_ORIGINAL_AUDIO_DIR / f"{track_id}.wav"
        if not original_audio_path.exists():
            original_audio_path_alt = GUITARSET_ORIGINAL_AUDIO_DIR / f"{track_id}_mic.wav"
            if not original_audio_path_alt.exists():
                print(f"  OSTRZEŻENIE: Nie znaleziono pliku audio dla {track_id}. Pomijanie kopiowania WAV.")
            else:
                original_audio_path = original_audio_path_alt

        if original_audio_path.exists():  # Sprawdź ponownie po ewentualnej zmianie na _alt
            output_wav_path = MIDI_OUTPUT_DIR / f"{track_id}.wav"
            try:
                shutil.copy(original_audio_path, output_wav_path)
                print(f"  Skopiowano WAV: {original_audio_path.name} -> {output_wav_path.name}")
            except Exception as e:
                print(f"  Błąd podczas kopiowania {original_audio_path.name}: {e}")
        else:
            print(f"  Nie można skopiować WAV dla {track_id}, plik źródłowy nie istnieje po obu próbach.")

        actual_length = feature_lengths_batch[i].item()

        notes_probs_sample = torch.sigmoid(output_logits_batch["notes"][i, :actual_length, :]).cpu().numpy()
        onsets_probs_sample = torch.sigmoid(output_logits_batch["onsets"][i, :actual_length, :]).cpu().numpy()

        # Użycie `optimal_thresholds`
        notes_binary_sample = (notes_probs_sample >= optimal_thresholds.get('notes', 0.5)).astype(int)
        onsets_binary_sample = (onsets_probs_sample >= optimal_thresholds.get('onsets', 0.5)).astype(int)

        output_mid_path = MIDI_OUTPUT_DIR / f"{track_id}.mid"
        predictions_to_midi(
            notes_binary=notes_binary_sample,
            onsets_binary=onsets_binary_sample,
            output_midi_path=str(output_mid_path),
            annotation_hop=ANNOTATION_HOP,
            notes_bins_per_semitone=NOTES_BINS_PER_SEMITONE,
            guitar_base_frequency=GUITAR_BASE_FREQUENCY
        )
        transcribed_count += 1

print(f"\nZakończono transkrypcję. Pliki .wav i .mid znajdują się w: {MIDI_OUTPUT_DIR}")


## 3.1. Testowanie modelu na własnych plikach

In [ ]:
from v1.own_files_test import transcribe_custom_audio_directory

CUSTOM_AUDIO_INPUT_DIR = "v1/test_data"
CUSTOM_MIDI_OUTPUT_BASE_DIR = "v1/transcribed_custom"
NUM_CUSTOM_SAMPLES_TO_TRANSCRIBE = 4  # Ile plików z tego katalogu przetworzyć (None dla wszystkich)

# Wywołanie funkcji transkrybującej
transcribe_custom_audio_directory(
    custom_audio_dir=CUSTOM_AUDIO_INPUT_DIR,
    model=model,
    device=device,
    optimal_thresholds_dict=optimal_thresholds,  # Użyj obliczonych progów
    output_base_dir=CUSTOM_MIDI_OUTPUT_BASE_DIR,
    num_files_to_process=NUM_CUSTOM_SAMPLES_TO_TRANSCRIBE,
    copy_wav=True  # Czy kopiować oryginalne .wav do katalogu wyjściowego
)